# 07 y 08 · Limpieza: Afiliaciones e Informe de Renovaciones (últimos 2 meses)

Dos reportes exportados de Zoho CRM el 13 de noviembre de 2026. Se limpian juntos porque tienen el mismo formato.

| | Entrada (`datos/originales/`) | Salida (`datos/limpios/`) | Una fila = |
|---|---|---|---|
| 07 | `Afiliaciones - Ultimos 2 Meses_anonimizado.xlsx` | `07_afiliaciones_limpio.csv` | una afiliación nueva cerrada |
| 08 | `Informe Renovaciones - Ultimos 2 Meses_anonimizado.xlsx` | `08_renovaciones_limpio.csv` | una renovación facturada |

Las columnas de dinero (`Importe`, `Valor de la Membresía`) vienen vacías por la anonimización y no se usan.

In [ ]:
import re
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Cómo viene un reporte agrupado de Zoho
No es una tabla normal. Las filas 0 a 5 son títulos (nombre del reporte, fecha, `Cantidad de registros : 14`) y la fila 6 tiene los encabezados.
Debajo, los datos vienen **agrupados**: el mes aparece solo en la primera fila del grupo, el KAM solo en la primera fila de su subgrupo, etc. Las demás filas tienen esas celdas vacías. Ejemplo:

| Mes | KAM | Empresa |
|---|---|---|
| Agosto 2026 ( 6 ) | KAM A ( 2 ) | EMP-0992 |
| | | EMP-0985 ← también es de Agosto y del KAM A |

Además, cada etiqueta de grupo trae entre paréntesis cuántos registros tiene: `( 6 )`. Al final hay una fila `Grand Total` y muchas filas vacías de separación.

**Cómo se limpia:** se rellena cada celda vacía con el valor de la fila de arriba (`ffill`), se dejan solo las filas que tienen empresa y se quitan los `( n )`.
Como dos columnas del reporte de afiliaciones se llaman igual, se trabaja por posición y se comprueba que los encabezados sean los esperados.
Antes se lee el número de registros que dice el reporte, para comprobar al final que no se perdió ni se inventó ninguno.

In [ ]:
def leer_reporte_zoho(archivo, encabezados, columnas_grupo):
    crudo = pd.read_excel(ORIGINALES / archivo, header=None)
    esperados = int(re.search(r"Cantidad de registros : (\d+)", crudo.iloc[4, 0]).group(1))
    assert crudo.iloc[6].tolist() == encabezados, crudo.iloc[6].tolist()  # el reporte no cambió de forma

    datos = crudo.iloc[7:].copy()
    datos = datos[datos[0] != "Grand Total"]
    datos[columnas_grupo] = datos[columnas_grupo].ffill()  # cada fila hereda su grupo
    return datos, esperados


def quitar_conteo(serie):
    """'Agosto 2026 ( 6 )' -> 'Agosto 2026'"""
    return serie.str.replace(r"\s*\(\s*\d+\s*\)\s*$", "", regex=True).str.strip()


MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
         "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12}
MESES_LARGOS = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
                "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}


def mes_a_fecha(serie):
    """'Agosto 2026' -> 2026-08-01 (Looker Studio agrupa mejor por fecha que por texto)"""
    partes = serie.str.lower().str.split()
    return pd.to_datetime(dict(year=partes.str[1].astype(int), month=partes.str[0].map(MESES_LARGOS), day=1))

## 2. Afiliaciones (07)
Grupos: mes de cierre → KAM (asesor comercial) → fecha de cierre. La fecha viene en español y abreviada (`ago 26, 2026`), así que se traduce el mes.

In [ ]:
ENCABEZADOS_AF = ["Fecha del cierre Negociacion", "Propietario de Trato [anonimizado]", "Fecha del cierre Negociacion",
                  "Nombre de Trato [anonimizado]", "Categoría de Membresía", "Importe [eliminado]"]
af, esperados_af = leer_reporte_zoho("Afiliaciones - Ultimos 2 Meses_anonimizado.xlsx", ENCABEZADOS_AF, columnas_grupo=[0, 1, 2])
af.columns = ["mes_cierre", "kam", "fecha_cierre", "id_empresa", "categoria_membresia", "importe"]
af = af[af["id_empresa"].notna()].drop(columns="importe")

for c in ["mes_cierre", "kam", "fecha_cierre"]:
    af[c] = quitar_conteo(af[c])
af["id_empresa"] = af["id_empresa"].str.strip()

fecha = af["fecha_cierre"].str.extract(r"^([a-z]{3}) (\d{1,2}), (\d{4})$")
assert fecha.notna().all().all(), af.loc[fecha.isna().any(axis=1), "fecha_cierre"]
af["fecha_cierre"] = pd.to_datetime(dict(year=fecha[2].astype(int), month=fecha[0].map(MESES),
                                         day=fecha[1].astype(int)))
af["mes_cierre"] = mes_a_fecha(af["mes_cierre"])
af = af.reset_index(drop=True)
af

Validaciones:
- Hay tantos registros como dice el reporte (14).
- La fecha de cierre cae en el mismo mes de su grupo (si no, el relleno hacia abajo habría mezclado grupos).
- Cada empresa aparece una vez y con el formato `EMP-0000`.

In [ ]:
assert len(af) == esperados_af == 14, (len(af), esperados_af)
assert (af["fecha_cierre"].dt.to_period("M") == af["mes_cierre"].dt.to_period("M")).all()
assert af["id_empresa"].is_unique and af["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert af["kam"].str.fullmatch(r"KAM [A-Z]").all()
assert set(af["categoria_membresia"]) <= {"Estándar", "Premium", "Personal"}
af.groupby(af["mes_cierre"].dt.strftime("%Y-%m")).size()

## 3. Renovaciones (08)
Grupos: mes de la factura de renovación → KAM. La fecha de expiración ya viene como fecha en el Excel, así que se usa directamente.

In [ ]:
ENCABEZADOS_REN = ["Fecha Factura Renovacion", "Propietario de Empresa [anonimizado]", "Nombre de Empresa [anonimizado]",
                   "Fecha de Expiración Afiliación", "Valor de la Membresía [eliminado]"]
ren, esperados_ren = leer_reporte_zoho("Informe Renovaciones - Ultimos 2 Meses_anonimizado.xlsx", ENCABEZADOS_REN, columnas_grupo=[0, 1])
ren.columns = ["mes_factura", "kam", "id_empresa", "fecha_expiracion", "valor_membresia"]
ren = ren[ren["id_empresa"].notna()].drop(columns="valor_membresia")

for c in ["mes_factura", "kam"]:
    ren[c] = quitar_conteo(ren[c])
ren["id_empresa"] = ren["id_empresa"].str.strip()
ren["fecha_expiracion"] = pd.to_datetime(ren["fecha_expiracion"])
ren["mes_factura"] = mes_a_fecha(ren["mes_factura"])
ren = ren.reset_index(drop=True)
ren

Validaciones:
- Hay tantos registros como dice el reporte (14).
- La afiliación renovada vence después del mes en que se facturó.
- Cada empresa aparece una vez y con el formato `EMP-0000`.

Nota: casi todas las renovaciones vencen en septiembre u octubre de 2027, pero dos de octubre vencen antes: EMP-0240 (31-12-2026) y EMP-0019 (31-05-2027). No es un error de la limpieza; pueden ser renovaciones parciales. Vale la pena preguntarle a ACTHO.

In [ ]:
assert len(ren) == esperados_ren == 14, (len(ren), esperados_ren)
assert ren["fecha_expiracion"].notna().all()
assert (ren["fecha_expiracion"] > ren["mes_factura"]).all()
assert ren["id_empresa"].is_unique and ren["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert ren["kam"].str.fullmatch(r"KAM [A-Z]").all()
ren.groupby(ren["mes_factura"].dt.strftime("%Y-%m")).size()

## 4. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Las dos tablas comparten `id_empresa` y `kam`, así que en Looker se pueden comparar afiliaciones y renovaciones por asesor.

In [ ]:
salida_af = LIMPIOS / "07_afiliaciones_limpio.csv"
salida_ren = LIMPIOS / "08_renovaciones_limpio.csv"
af.to_csv(salida_af, index=False, encoding="utf-8-sig")
ren.to_csv(salida_ren, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida_af).shape == (14, 5)
assert pd.read_csv(salida_ren).shape == (14, 4)
salida_af.name, salida_ren.name